# Chapter 10: The Full GPT Architecture

[Read this chapter online](https://jackluu.io/book/section-3-the-transformer/ch10-full-gpt-architecture/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch10-full-gpt-architecture.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 10: The Full GPT Architecture

![Where we are in the pipeline](../assets/diagrams/ch10-where-we-are.png){ width="756" }
*Figure 10.1: We connect everything to output a prediction.*

Now that we have built all the core components of a Transformer model in previous chapters, we can assemble them (Figure 10.1). In this chapter you will:

* Build the full GPT model from end to end.
* See how data flows through the entire pipeline.
* Understand the final raw scores, called logits.

**Words to Know**
    - **LM Head**: The final linear layer that maps internal numbers back to vocabulary words.
    - **Logits**: The raw scores the model outputs for each possible next token.

## Theory

### The Final Assembly

The complete GPT model is surprisingly simple once you have the pieces. 

![The pipeline flows from embeddings through blocks to the LM head](../assets/diagrams/ch10-stacked-architecture.png){ width="698" }
*Figure 10.2: The full forward pass of the GPT model.*

It flows exactly like a production assembly line (as shown in Figure 10.2):

1. Turn text into a list of numbers (Token IDs).
2. Look up the meaning and position vectors (Embeddings).
3. Process them through a stack of Transformer blocks.
4. Tidy up the numbers one last time (Final LayerNorm).
5. Map the internal 128 numbers back to the 65 possible characters (LM Head).

### Tracing the Flow

Let's trace a concrete example. Suppose we feed the model the word `"ROMEO"` (5 tokens). 

**Step 1: Input**
The input is `[30, 27, 25, 17, 27]`. These are the specific token IDs for the characters "ROMEO", where 'R' is 30, 'O' is 27, 'M' is 25, and 'E' is 17 according to our vocabulary.

**Step 2: Embeddings**
The model converts these 5 IDs into 5 rich vectors (128 numbers each), combining both meaning and position.

**Step 3: Transformer Blocks**
The data passes through 4 blocks. After 4 blocks, the token `O` (at the end) has looked at all the previous letters. Its 128 numbers now encode something like "I am the last letter of a name from a famous play."

**Step 4 & 5: Final Output**
A final LayerNorm stabilizes the numbers. Then, the **LM Head** (Language Model Head) takes those 128 numbers and projects them to 65 numbers. Why 65? Because our Shakespeare dataset has exactly 65 unique characters. 

![The internal 128 numbers are projected to 65 vocabulary scores](../assets/diagrams/ch10-lm-head.png){ width="618" }
*Figure 10.3: The LM Head projects internal data back to our vocabulary.*

As Figure 10.3 illustrates, the model outputs 65 scores for every single token in the sequence. Each position independently predicts the character that comes next. The highest score at the final position is the model's best guess for what comes after "ROMEO".

### What Are Logits?

The 65 scores the LM Head outputs are called **logits**. 

Logit is a technical term for "raw score" (Figure 10.4). These numbers can be negative, zero, or very large. They do not add up to 1, so they are not probabilities yet. In Chapter 11, we will convert these raw scores into proper percentages to generate text. For now, just know that a higher logit means the model is more confident in that character.

![Logit scores for the token O show highest confidence for N and I](../assets/diagrams/ch10-logits.png){ width="378" }
*Figure 10.4: Logits are the raw scores for the next character.*

Returning to the big picture map in Figure 10.1, we have built everything from Text to Next-Token Scores. The model is fully assembled, but right now it only outputs random guesses. We need to train it.

## Code

```python
class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        # Look up table for token vectors
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        # Look up table for position vectors
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)

        # A sequence of transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock() for _ in range(cfg.n_layers)
        ])

        # Final layer normalization and linear layer to output scores
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape
        assert T <= self.cfg.block_size, \
            f"Sequence length {T} exceeds block_size " \
            f"{self.cfg.block_size}"

        tok_emb = self.token_emb(token_ids)
        positions = torch.arange(T, device=token_ids.device)
        pos_emb = self.pos_emb(positions)

        # Combine token and position embeddings
        x = tok_emb + pos_emb

        # Pass through the transformer blocks
        x = self.blocks(x)

        # Final normalization and produce scores
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits
```

Run the script to see the parameter breakdown and a test run.

```python
$ python src/ch09_gpt_model.py
Model parameter breakdown:
  Token embedding  :      8,320
  Position embedding:    16,384
  4 Transformer blocks:   791,552
  LM head          :      8,320
  LayerNorm (final):        256
  ---
  TOTAL            :    824,832

Input  shape: torch.Size([2, 10])   (B, T)
Output shape: torch.Size([2, 10, 65])  (B, T, vocab_size)

At each position, model outputs 65 scores.
The highest score = best guess for next character.

First position logits (top 5 scores):
  token 26: 1.861
  token 21: 1.195
  token 44: 1.118
  token 57: 1.071
  token 34: 0.972

Note: these are random (untrained model).
```

**What just happened:**

1. Line 1 assembles the complete GPT class.
2. Line 12 creates the blocks that give us exactly 824,832 parameters (weights). The original GPT-2 Small had 117 million. Our model uses the exact same architecture, just scaled down.
3. Line 38 finishes the forward pass, where the model outputs 65 logits (scores) for each position.

**Shape Check:**

- Input token IDs: `[Batch, Time]`
- Output logits: `[Batch, Time, 65]` (Vocab size is 65)

## Try It

**Try It**
    Open `src/ch09_gpt_model.py` and modify `config.n_layers = 6` just to test. Run the script again. Watch how the parameter count increases. The blocks contain the vast majority of the model's "brain".

**In Business**
    Building the final GPT architecture is like assembling a complete production pipeline from modular components. In our house-style writing assistant, we combine a data reader (embeddings), an analysis engine (the blocks), and an output formatter (the LM head). Because it is modular, you can upgrade the engine (add more layers) without rewriting the rest of the pipeline.

## Key Takeaways

* The full model stacks embeddings, Transformer blocks, and a final LM head.
* The LM head is a linear layer that projects the internal representations back to the vocabulary.
* The model outputs logits, which are raw, un-normalized scores for the next token.
* Our complete model has ~825K parameters, proving that powerful architectures can be built at a small scale.

## Check Your Understanding

1. If the input sequence has 10 tokens, how many predictions does the model make?
2. Why does the LM head output exactly 65 numbers per token?
3. What is a logit?


## Further Reading

**One model, many jobs, no retraining.** The question was whether a model trained only to predict the next word would pick up skills nobody trained it for. Trained on a large, varied sweep of web pages, it began answering questions, summarizing and translating with no task-specific training at all, simply because the prompt made the task clear. That settled the architecture question for text generation: a decoder that predicts the next token, which is the model in Chapter 10.

<div class="refs" markdown>

Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language models are unsupervised multitask learners* [Technical report]. OpenAI. https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

</div>

---

### `src/ch09_gpt_model.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch09_gpt_model.py"   # a cell has none, and the file uses it to find the text

"""
Construct the full GPT architecture.
This file belongs to Chapter 10.
Run: python src/ch09_gpt_model.py
"""
import torch
import torch.nn as nn
import os
import sys

from src.utils.config import GPTConfig
from src.ch08_transformer_block import TransformerBlock

# Settings
config = GPTConfig()

# --- The Idea ---
class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        # Look up table for token vectors
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        # Look up table for position vectors
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)

        # A sequence of transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock() for _ in range(cfg.n_layers)
        ])

        # Final layer normalization and linear layer to output scores
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape
        assert T <= self.cfg.block_size, \
            f"Sequence length {T} exceeds block_size " \
            f"{self.cfg.block_size}"

        tok_emb = self.token_emb(token_ids)
        positions = torch.arange(T, device=token_ids.device)
        pos_emb = self.pos_emb(positions)

        # Combine token and position embeddings
        x = tok_emb + pos_emb

        # Pass through the transformer blocks
        x = self.blocks(x)

        # Final normalization and produce scores
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 10: The Full GPT Model\n")

    model = GPT(config)
    total_params = sum(p.numel() for p in model.parameters())

    print("Model parameter breakdown:")
    print(f"  Token embedding  : {model.token_emb.weight.numel():>10,}")
    print(f"  Position embedding: {model.pos_emb.weight.numel():>9,}")
    block_params = sum(p.numel() for p in model.blocks.parameters())
    print(f"  {config.n_layers} Transformer blocks: {block_params:>9,}")
    lm_head_params = sum(p.numel() for p in model.lm_head.parameters())
    print(f"  LM head          : {lm_head_params:>10,}")
    ln_f_params = sum(p.numel() for p in model.ln_f.parameters())
    print(f"  LayerNorm (final): {ln_f_params:>10,}")
    print("  ---")
    print(f"  TOTAL            : {total_params:>10,}")

    B, T = 2, 10
    token_ids = torch.randint(0, config.vocab_size, (B, T))
    logits = model(token_ids)

    print(f"\nInput  shape: {token_ids.shape}   (B, T)")
    print(f"Output shape: {logits.shape}  (B, T, vocab_size)")
    print(f"\nAt each position, model outputs {config.vocab_size} scores.")
    print("The highest score = best guess for next character.")

    first_pos_logits = logits[0, 0]
    print("\nFirst position logits (top 5 scores):")
    top5 = first_pos_logits.topk(5)
    for score, idx in zip(top5.values.tolist(), top5.indices.tolist()):
        print(f"  token {idx:2d}: {score:.3f}")

    print("\nNote: these are random (untrained model).")
    print("\nFull GPT model done! Ready for Chapter 11.")